# Experiment 9 — Retrieval-Augmented Generation (RAG) Document QA
**Aim:** Retrieve evidence from BattingCoach project documentation and pass it to a language model for grounded question answering.


# Data source and reproducibility
The BattingCoach repository contains a Roboflow Cricket Bat dataset (930 images, YOLOv8-OBB format; class `Bat`) and a separate 10-class cricket-shot video classification dataset referenced by the project code (`rokmr/cricketshot` / deployed VideoMAE checkpoint). The repository README documents the wider MediaPipe + YOLO + VideoMAE system.

This notebook uses the project data where it is technically appropriate. For experiments whose prescribed task is unrelated to cricket images (for example MNIST VAE/GAN or English→French translation), it uses the benchmark dataset specified in the experiment sheet and explains why. Dataset downloads can require internet access and, for some Hugging Face datasets, account/token permissions.

**Repository:** https://github.com/Arnavch2024/BattingCoach
**Roboflow dataset source:** https://universe.roboflow.com/mohammad-shafay-qgjsx/cricket-bat-dataset-ln3te
**Project shot classifier dataset identifier:** `rokmr/cricketshot`


### Dataset mapping
The selected corpus consists of BattingCoach's documented architecture, dataset description, biomechanics, shot-class names, and practice modes. These statements are taken from the repository's README and dataset metadata/code descriptions. For a submission, cite the repository pages/files as your document sources and capture the exact retrieved chunks produced by this notebook.

## 1. Install retrieval and generation dependencies

In [1]:
from importlib.metadata import version

for package in ('sentence-transformers', 'faiss-cpu', 'transformers', 'torch'):
    try:
        print(f"{package}: {version(package)}")
    except Exception as exc:
        print(f"{package}: unavailable ({exc})")


sentence-transformers: 6.2.0
faiss-cpu: 1.15.1
transformers: 5.22.1
torch: 2.15.0


## 2. Build a source-labelled document corpus

In [2]:
docs=[
("README.md#overview","BatCoach AI Pro performs 3D joint estimation and kinematic angle extraction using Google MediaPipe."),
("README.md#dual-practice-modes","Shadow Practice (no_bat) suppresses YOLO bat checks and uses MediaPipe pose estimation and VideoMAE classification."),
("README.md#dual-practice-modes","Live Willow Practice (with_bat) uses YOLOv8-OBB bat detection for blade angle and bat-pad gap analysis."),
("README.dataset.txt","The Cricket Bat Dataset is hosted on Roboflow Universe and is identified as CC BY 4.0."),
("README.roboflow.txt","The dataset contains 930 images with bat annotations in YOLOv8 Oriented Object Detection format."),
("data.yaml","The dataset defines train, valid and test image splits, with class 0 named Bat."),
("coach_backend.py#model","The backend defines ten stroke classes: cover, defense, flick, hook, late_cut, lofted, pull, square_cut, straight and sweep."),
("biomechanics.py","The biomechanics engine computes 3D angles and evaluates batting metrics such as elbow, knee, balance and spine alignment."),
("README.md#architecture","The backend uses FastAPI and WebSockets for interactive, real-time coaching."),
("README.md#deployment","The application architecture includes a Next.js frontend, Azure Container Apps backend and Supabase PostgreSQL persistence.")
]

## 3. Chunk and embed source documents

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss
embedder=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts=[d[1] for d in docs]
emb=embedder.encode(texts,convert_to_numpy=True,normalize_embeddings=True).astype("float32")
index=faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
print("Chunks:",len(texts),"| vector dimension:",emb.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks: 10 | vector dimension: 384


## 4. Retriever and context builder

In [4]:
def retrieve(question,k=3):
    q=embedder.encode([question],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    scores,ids=index.search(q,k)
    return [{"source":docs[i][0],"text":docs[i][1],"similarity":float(s)}
            for s,i in zip(scores[0],ids[0])]
def build_prompt(question,hits):
    context="\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
    return f'''You are a careful assistant. Answer only from the context below.
If the answer is absent, say that the repository context does not provide enough information.
Do not invent facts. Cite source labels in square brackets.
CONTEXT:
{context}
QUESTION: {question}
ANSWER:'''

## 5. Load an optional small instruction model
If the runtime cannot fit the model, use the retrieved context and prompt as a complete retrieval demonstration; do not fabricate model-generated answers.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
try:
    tok=AutoTokenizer.from_pretrained(MODEL_ID)
    llm=AutoModelForCausalLM.from_pretrained(
        MODEL_ID,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto")
    def generate_answer(prompt):
        chat=tok.apply_chat_template([{"role":"user","content":prompt}],
                                    tokenize=False,add_generation_prompt=True)
        inputs=tok(chat,return_tensors="pt").to(llm.device)
        with torch.no_grad():
            out=llm.generate(**inputs,max_new_tokens=180,do_sample=False,
                             pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)
    LLM_AVAILABLE=True
    print("Loaded:",MODEL_ID)
except Exception as exc:
    LLM_AVAILABLE=False
    print("LLM unavailable in this runtime:",type(exc).__name__,str(exc)[:300])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct


## 6. Ask six test questions

In [6]:
questions=[
"What technology is used to estimate the batter's 3D joint positions?",
"How does the system behave during shadow practice?",
"How is the cricket bat detected during live willow practice?",
"Which ten stroke classes are supported by the VideoMAE classifier?",
"What is the dataset annotation format and how many images does it contain?",
"What is the exact monthly subscription fee charged by BatCoach?"
]
rag_results=[]
for q in questions:
    hits=retrieve(q,3)
    prompt=build_prompt(q,hits)
    answer=generate_answer(prompt) if LLM_AVAILABLE else "LLM unavailable: inspect the retrieved context and answer manually."
    rag_results.append({"question":q,"answer":answer,"sources":[h["source"] for h in hits],"hits":hits})
    print("\nQUESTION:",q,"\nANSWER:",answer,"\nSOURCES:",[h["source"] for h in hits])


QUESTION: What technology is used to estimate the batter's 3D joint positions? 
ANSWER: The biomechanics engine uses Google MediaPipe to compute 3D angles and evaluate batting metrics such as elbow, knee, balance, and spine alignment. 
SOURCES: ['biomechanics.py', 'README.md#overview', 'README.md#dual-practice-modes']

QUESTION: How does the system behave during shadow practice? 
ANSWER: The system behaves differently during shadow practice compared to regular practice. During shadow practice, it employs dual practices with two modes:

1. **No_Bat Mode**: This mode disables YOLO (You Only Look Once) detection for bounding boxes, which helps in reducing false positives and improving accuracy.
2. **MediaPipe Pose Estimation Mode**: This mode utilizes MediaPipe's pose estimation capabilities to detect poses accurately, enhancing the system's ability to recognize different types of movements.

Additionally, the system also includes VideoMAE (Video Motion Average Error) classification, whi

## 7. Compare retrieval-only evidence and answer grounding

In [7]:
for item in rag_results:
    print("\nQ:",item["question"])
    for h in item["hits"]:
        print(round(h["similarity"],3),h["source"],h["text"])
    print("Generated answer:",item["answer"])


Q: What technology is used to estimate the batter's 3D joint positions?
0.686 biomechanics.py The biomechanics engine computes 3D angles and evaluates batting metrics such as elbow, knee, balance and spine alignment.
0.632 README.md#overview BatCoach AI Pro performs 3D joint estimation and kinematic angle extraction using Google MediaPipe.
0.378 README.md#dual-practice-modes Live Willow Practice (with_bat) uses YOLOv8-OBB bat detection for blade angle and bat-pad gap analysis.
Generated answer: The biomechanics engine uses Google MediaPipe to compute 3D angles and evaluate batting metrics such as elbow, knee, balance, and spine alignment.

Q: How does the system behave during shadow practice?
0.53 README.md#dual-practice-modes Shadow Practice (no_bat) suppresses YOLO bat checks and uses MediaPipe pose estimation and VideoMAE classification.
0.137 data.yaml The dataset defines train, valid and test image splits, with class 0 named Bat.
0.121 coach_backend.py#model The backend defines t

## 8. Evaluation table — complete after reviewing real outputs

In [8]:
import re
import pandas as pd

def answer_terms(text):
    return {t for t in re.findall(r"[a-z0-9]+", text.lower()) if len(t) > 4}
eval_rows=[]
for item in rag_results:
    context = " ".join(h["text"] for h in item["hits"])
    question_terms = answer_terms(item["question"])
    retrieved_terms = answer_terms(context)
    answer_terms_set = answer_terms(item["answer"])
    relevant = bool(question_terms & retrieved_terms) and max(h["similarity"] for h in item["hits"]) >= 0.25
    supported = bool(answer_terms_set & retrieved_terms) if LLM_AVAILABLE else False
    eval_rows.append({
        "Question": item["question"],
        "Top-k relevant?": "Yes" if relevant else "No",
        "Answer supported?": "Yes" if supported else "Review",
        "Sources": ", ".join(item["sources"]),
        "Hallucination?": "No evidence" if supported else "Potential unsupported answer",
        "Observation": f"max similarity={max(h['similarity'] for h in item['hits']):.3f}"
    })
evaluation_table = pd.DataFrame(eval_rows)
evaluation_table

# Compare the grounded answer with a direct answer using the same model and question.
comparison=[]
for item in rag_results:
    direct = generate_answer(item["question"]) if LLM_AVAILABLE else "LLM unavailable"
    comparison.append({"Question":item["question"], "Direct answer":direct, "RAG answer":item["answer"], "Retrieved sources":", ".join(item["sources"])})
comparison_table=pd.DataFrame(comparison)
comparison_table[["Question","Retrieved sources"]]


,Question,Retrieved sources
0,What technology is used to estimate the batter...,"biomechanics.py, README.md#overview, README.md..."
1,How does the system behave during shadow pract...,"README.md#dual-practice-modes, data.yaml, coac..."
2,How is the cricket bat detected during live wi...,"README.md#dual-practice-modes, README.dataset...."
3,Which ten stroke classes are supported by the ...,"coach_backend.py#model, data.yaml, README.robo..."
4,What is the dataset annotation format and how ...,"data.yaml, README.roboflow.txt, README.dataset..."
5,What is the exact monthly subscription fee cha...,"README.dataset.txt, README.md#dual-practice-mo..."


## 9. Conclusion
Evaluate retrieval relevance separately from answer quality. Confirm that the final pricing question is not answered with invented detail. Note whether the system admits missing information and provides source traceability.